# Initial Exploration — FlyWire FAFB v783 connectome

Exploratory data analysis of the stage-1 artifacts (built by `flyconn.data_prep`).

**Kernel:** `Python (flyconn_eda)`  ·  **Data:** `$FLYCONN_DATA_ROOT/v783/processed/`

Sections:
1. Load + sanity
2. Composition (super_class, cell_class, side, flow)
3. Neurotransmitters
4. Degree distributions
5. Hub neurons
6. Signed adjacency structure
7. Connectivity between super-classes

Everything loads from the canonical thresholded connectome (`edges.parquet`, ≥5 synapses).
The no-threshold full graph (`edges_full.parquet`) and unsigned/full matrices are available
via the same `paths` object if you want them.

## 1. Load + sanity

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

from flyconn.config import load_config
from flyconn.io import read_parquet, load_csr

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)

# Resolve artifact paths (uses FLYCONN_DATA_ROOT; defaults to the scratch root).
cfg = load_config()
P = cfg.paths()
print("data root:", P.root)
print("nt policy :", cfg.default_nt_policy, "| synapse threshold:", cfg.synapse_threshold)

In [ ]:
# Node table (row i == node idx i) and the canonical thresholded edge list.
neurons = read_parquet(P.neurons)
edges = read_parquet(P.edges)
N = len(neurons)

# Signed adjacency (default policy) + unsigned counts. Source-major: A[i, j] = i -> j.
A_signed = load_csr(P.adjacency_signed(cfg.default_nt_policy))
A_counts = load_csr(P.adjacency_counts)

print(f"neurons: {N:,}   edges (>= {cfg.synapse_threshold} syn): {len(edges):,}")
print(f"A_signed: {A_signed.shape}  nnz={A_signed.nnz:,}")
assert A_signed.shape == (N, N)
assert edges['pre_idx'].between(0, N - 1).all() and edges['post_idx'].between(0, N - 1).all()
neurons.head()

## 2. Composition

What kinds of neurons make up the brain — by super-class, side, and information flow.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sc = neurons['super_class'].value_counts()
sns.barplot(x=sc.values, y=sc.index, ax=axes[0], color='steelblue')
axes[0].set(title='Neurons per super_class', xlabel='count', ylabel='')

side = neurons['side'].value_counts(dropna=False)
axes[1].pie(side.values, labels=[str(s) for s in side.index], autopct='%1.0f%%', startangle=90)
axes[1].set(title='Soma side (L/R symmetry check)')

flow = neurons['flow'].value_counts(dropna=False)
sns.barplot(x=flow.values, y=[str(s) for s in flow.index], ax=axes[2], color='seagreen')
axes[2].set(title='Flow (afferent / intrinsic / efferent)', xlabel='count', ylabel='')

plt.tight_layout(); plt.show()

In [ ]:
# Top cell classes and cell types
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, col, n in zip(axes, ['cell_class', 'cell_type'], [20, 25]):
    top = neurons[col].value_counts().head(n)
    sns.barplot(x=top.values, y=top.index, ax=ax, color='indianred')
    ax.set(title=f'Top {n} {col}', xlabel='count', ylabel='')
plt.tight_layout(); plt.show()

## 3. Neurotransmitters

`nt_canonical` is the normalized predicted transmitter per neuron (the basis for the
signed adjacency). Recall **glutamate is inhibitory** in the fly under the default policy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

nt = neurons['nt_canonical'].value_counts(dropna=False)
sns.barplot(x=nt.values, y=[str(s) for s in nt.index], ax=axes[0], color='slateblue')
axes[0].set(title='Neurons per predicted neurotransmitter', xlabel='count', ylabel='')

# NT prediction confidence distribution
sns.histplot(neurons['top_nt_conf'].dropna(), bins=40, ax=axes[1], color='slateblue')
axes[1].set(title='top_nt prediction confidence', xlabel='confidence')
plt.tight_layout(); plt.show()

In [ ]:
# NT composition within each super_class (stacked proportions)
ct = pd.crosstab(neurons['super_class'], neurons['nt_canonical'], normalize='index')
ct = ct.loc[neurons['super_class'].value_counts().index]  # order by size
ct.plot(kind='barh', stacked=True, figsize=(11, 6), colormap='tab10')
plt.title('Neurotransmitter composition by super_class')
plt.xlabel('proportion'); plt.legend(title='NT', bbox_to_anchor=(1.01, 1)); plt.tight_layout(); plt.show()

## 4. Degree distributions

Out-degree = number of distinct downstream partners; in-degree = upstream partners.
Synaptic strength uses the raw synapse counts.

In [ ]:
# Partner counts (binary degree) and synaptic strength (weighted), from the counts matrix.
Abin = (A_counts != 0)
out_deg = np.asarray(Abin.sum(axis=1)).ravel()   # # downstream partners
in_deg = np.asarray(Abin.sum(axis=0)).ravel()    # # upstream partners
out_str = np.asarray(A_counts.sum(axis=1)).ravel()  # total outgoing synapses
in_str = np.asarray(A_counts.sum(axis=0)).ravel()   # total incoming synapses
neurons = neurons.assign(out_deg=out_deg, in_deg=in_deg, out_str=out_str, in_str=in_str)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for deg, lab, c in [(out_deg, 'out-degree', 'C0'), (in_deg, 'in-degree', 'C3')]:
    vals = deg[deg > 0]
    axes[0].hist(vals, bins=np.logspace(0, np.log10(vals.max()), 50),
                 histtype='step', linewidth=2, label=lab, color=c)
axes[0].set(xscale='log', yscale='log', xlabel='degree', ylabel='# neurons',
            title='Degree distribution (log-log)')
axes[0].legend()

axes[1].scatter(out_deg, in_deg, s=3, alpha=0.15)
axes[1].set(xscale='symlog', yscale='symlog', xlabel='out-degree', ylabel='in-degree',
            title='In- vs out-degree per neuron')
plt.tight_layout(); plt.show()

## 5. Hub neurons

The most broadly-connected neurons. CT1 and APL (giant GABAergic interneurons) should top the list.

In [ ]:
cols = ['idx', 'root_id', 'cell_type', 'super_class', 'side', 'nt_canonical',
        'out_deg', 'in_deg', 'out_str', 'in_str']
print('Top 15 by out-degree (broadcasters):')
display(neurons.nlargest(15, 'out_deg')[cols])
print('Top 15 by in-degree (integrators):')
display(neurons.nlargest(15, 'in_deg')[cols])

## 6. Signed adjacency structure

Under the default `flyvis_standard` policy: ACh excitatory (+), GABA & Glut inhibitory (−).

In [ ]:
w = A_signed.data
pos, neg = w[w > 0], w[w < 0]
print(f"edges with sign: {len(w):,}   (+){len(pos):,}  (-){len(neg):,}")
print(f"excitatory mass: {pos.sum():,.0f}   inhibitory mass: {neg.sum():,.0f}")
print(f"E:I weight ratio: {pos.sum() / -neg.sum():.2f}")

fig, ax = plt.subplots(figsize=(9, 5))
lim = np.percentile(np.abs(w), 99)
ax.hist(np.clip(w, -lim, lim), bins=80, color='gray')
ax.axvline(0, color='k', lw=1)
ax.set(title='Signed edge weights (clipped at 99th pct)', xlabel='signed synapse count',
       yscale='log', ylabel='# edges')
plt.tight_layout(); plt.show()

## 7. Connectivity between super-classes

Total synaptic flow from each presynaptic super-class to each postsynaptic super-class.

In [ ]:
# Map each edge's endpoints to super_class, then sum synapses per (pre_sc -> post_sc).
sc = neurons['super_class'].fillna('unknown').to_numpy()
e = edges.copy()
e['pre_sc'] = sc[e['pre_idx'].to_numpy()]
e['post_sc'] = sc[e['post_idx'].to_numpy()]
flow = e.groupby(['pre_sc', 'post_sc'])['syn_count'].sum().unstack(fill_value=0)

order = neurons['super_class'].value_counts().index.tolist()
flow = flow.reindex(index=order, columns=order, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(np.log10(flow + 1), annot=False, cmap='magma',
            cbar_kws={'label': 'log10(synapses + 1)'}, ax=ax)
ax.set(title='Synaptic flow: presynaptic (row) -> postsynaptic (col) super_class',
       xlabel='postsynaptic super_class', ylabel='presynaptic super_class')
plt.tight_layout(); plt.show()

---
### Where to go next
- `edges_full.parquet` / `P.adjacency_counts_full` — the no-threshold full graph.
- `P.adjacency_signed('glut_excitatory')` — compare the alternative NT-sign policy.
- `networkx` / `scipy.sparse.csgraph` — components, paths, motifs (stage 4).
- Spatial: `soma_x/y/z` columns for anatomical plots.
- Modeling (stage 3): `torch.load(P.adjacency_pt)` gives the model-ready sparse tensor;
  transpose for an RNN weight matrix `W[post, pre]`.

---
# Part II — Toward a cell-type-level **visual** ANN

The sections above explored the whole brain. Here we narrow to the **visual system**
and reshape it into the substrate for a trainable network:

> **Target model (first cut):** one node per **cell type** (collapsing the ~721
> repeated retinotopic columns into a single shared node), edges = aggregated
> **type → type** synapse weights. Input = photoreceptors (R1-6, R7, R8); readout =
> visual-projection neurons (LC*, MeTu*, …). This is a deliberately coarser cousin of
> flyvis (Lappalainen 2024): the type→type weight matrix *is* the column-shared kernel,
> so we need no retinotopy/column assignments to get started.

We build two small helpers, then six diagnostic sections that reveal the structure the
model will inherit.

## A. Visual subset + helpers

In [ ]:
# Visual super-classes + photoreceptors (which live under 'sensory').
VISUAL_SUPERCLASSES = ['optic', 'visual_projection', 'visual_centrifugal']
PHOTORECEPTOR_TYPES = ['R1-6', 'R7', 'R8']           # input cells (in 'sensory')

def visual_mask(neurons):
    # Boolean mask: optic/VP/VC neurons + photoreceptors from sensory.
    in_vis_sc = neurons['super_class'].isin(VISUAL_SUPERCLASSES)
    is_photo = ((neurons['super_class'] == 'sensory')
                & neurons['cell_type'].isin(PHOTORECEPTOR_TYPES))
    return in_vis_sc | is_photo

def collapse_to_celltype(neurons, edges, key='cell_type', signed=False, drop_null=True):
    # Collapse a neuron-level graph to a TYPE x TYPE weighted matrix.
    # Returns (types, T, counts):
    #   types  : sorted list of type labels (row/col order of T)
    #   T      : (K,K) ndarray, T[a,b] = synapses from type a -> type b
    #            (signed by the presynaptic type's NT if signed=True)
    #   counts : #neurons per type (for normalization)
    # Only edges with BOTH endpoints inside `neurons` are used.
    idx2type = neurons.set_index('idx')[key]
    e = edges.copy()
    e['pre_t'] = e['pre_idx'].map(idx2type)
    e['post_t'] = e['post_idx'].map(idx2type)
    e = e.dropna(subset=['pre_t', 'post_t'])         # keep edges inside the subset
    if drop_null:
        e = e[(e['pre_t'] != 'nan') & (e['post_t'] != 'nan')]
    w = e['syn_count'].astype(float)
    if signed:
        # sign from the presynaptic neuron's canonical NT (flyvis_standard policy)
        sign_map = {'acetylcholine': 1, 'gaba': -1, 'glutamate': -1,
                    'octopamine': 0, 'serotonin': 0, 'dopamine': 0}
        s = e['pre_nt'].map(sign_map).fillna(0).to_numpy()
        w = w * s
    g = (pd.DataFrame({'pre_t': e['pre_t'], 'post_t': e['post_t'], 'w': w})
         .groupby(['pre_t', 'post_t'])['w'].sum().reset_index())
    types = sorted(set(g['pre_t']) | set(g['post_t']))
    ti = {t: i for i, t in enumerate(types)}
    K = len(types)
    T = np.zeros((K, K), dtype=float)
    T[g['pre_t'].map(ti).to_numpy(), g['post_t'].map(ti).to_numpy()] = g['w'].to_numpy()
    counts = neurons[key].value_counts().reindex(types).fillna(0).astype(int)
    return types, T, counts

print("helpers defined: visual_mask(), collapse_to_celltype()")

In [ ]:
# Build the visual subset (neuron-level) and report composition.
vis = neurons[visual_mask(neurons)].copy()
print(f"visual neurons: {len(vis):,}  ({len(vis)/len(neurons):.0%} of brain)")
print(f"unique cell_type: {vis['cell_type'].nunique():,}   "
      f"(null cell_type: {int(vis['cell_type'].isna().sum())})")
print("\nby super_class:")
print(vis['super_class'].value_counts())

# Input set (photoreceptors) and candidate readout set (visual projection types)
photo_types = sorted(t for t in vis['cell_type'].dropna().unique() if t in PHOTORECEPTOR_TYPES)
vpn_types = sorted(vis.loc[vis['super_class']=='visual_projection','cell_type'].dropna().unique())
print(f"\nINPUT types (photoreceptors): {photo_types}")
print(f"READOUT candidates (visual_projection types): {len(vpn_types)} types, "
      f"e.g. {vpn_types[:12]}")

## B. Cell-type connectivity matrix

The core object: a **type × type** weighted adjacency `T` where `T[a,b]` is the total
synapses from type-a neurons to type-b neurons. We show it three ways:
- **raw** (total synapses),
- **per-source-neuron normalized** (`T[a,b] / n_neurons(a)` — the natural scale for a
  shared weight, since each type has ~one neuron per column),
- **signed** (E/I via the presynaptic NT) — this is literally the shape of the model's
  weight matrix.

With 680 types the full matrix is unreadable, so we focus the heatmaps on the **top-K
most populous types** while computing stats on the full matrix.

In [ ]:
types_all, T_raw, counts = collapse_to_celltype(vis, edges, signed=False)
_, T_sgn, _ = collapse_to_celltype(vis, edges, signed=True)
K = len(types_all)
print(f"type-collapsed matrix: {K} x {K} types, "
      f"{(T_raw!=0).sum():,} nonzero type->type connections")
print(f"total synapses inside visual subset: {T_raw.sum():,.0f}")

# per-source-neuron normalized
nrm = counts.to_numpy().astype(float); nrm[nrm == 0] = 1
T_norm = T_raw / nrm[:, None]

# focus heatmaps on the TOP-K populous types
TOPK = 60
top_types = counts.sort_values(ascending=False).head(TOPK).index.tolist()
ti = {t: i for i, t in enumerate(types_all)}
sel = [ti[t] for t in top_types]

import matplotlib.colors as mcolors
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
sns.heatmap(np.log10(T_raw[np.ix_(sel, sel)] + 1), ax=axes[0], cmap='magma',
            xticklabels=top_types, yticklabels=top_types, cbar_kws={'label': 'log10(syn+1)'})
axes[0].set(title=f'Type->type synapses (raw, top {TOPK})',
            xlabel='postsynaptic type', ylabel='presynaptic type')
Tn = T_norm[np.ix_(sel, sel)]
sns.heatmap(Tn, ax=axes[1], cmap='magma',
            xticklabels=top_types, yticklabels=top_types,
            cbar_kws={'label': 'mean syn / source neuron'})
axes[1].set(title=f'Type->type (per-source-neuron normalized, top {TOPK})',
            xlabel='postsynaptic type', ylabel='presynaptic type')
for ax in axes:
    ax.tick_params(labelsize=6)
plt.tight_layout(); plt.show()

In [ ]:
# Signed version: red = inhibitory, blue = excitatory (this IS the weight-matrix shape)
Ts = T_sgn[np.ix_(sel, sel)]
lim = np.percentile(np.abs(Ts[Ts != 0]), 98) if (Ts != 0).any() else 1
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(np.clip(Ts, -lim, lim), ax=ax, cmap='RdBu_r', center=0,
            xticklabels=top_types, yticklabels=top_types,
            cbar_kws={'label': 'signed synapses (clipped)'})
ax.set(title=f'Signed type->type weights (flyvis_standard; top {TOPK})',
       xlabel='postsynaptic type', ylabel='presynaptic type')
ax.tick_params(labelsize=6)
plt.tight_layout(); plt.show()
exc = T_sgn[T_sgn > 0].sum(); inh = T_sgn[T_sgn < 0].sum()
print(f"excitatory mass: {exc:,.0f}   inhibitory mass: {inh:,.0f}   E:I = {exc/-inh:.2f}")

## C. Layered / feedforward structure

The fly visual system is a feedforward stack: **retina → lamina (L) → medulla (Mi/Tm/C)
→ lobula & lobula plate (T4/T5, LC) → central brain (VPN)**. We assign each type a coarse
**depth** from its region tag (`cell_class`) and known role, reorder the matrix by depth,
and quantify how much synaptic mass flows *forward* (lower→higher depth) vs *back*.

In [ ]:
def type_depth(row):
    # Coarse feedforward depth for a (type, super_class, cell_class) row.
    ct = str(row['cell_type']); sc = str(row['super_class']); cc = str(row['cell_class'])
    if ct in PHOTORECEPTOR_TYPES:           return 0     # retina
    if ct.startswith('L') and ct[1:2].isdigit(): return 1  # lamina monopolar L1..L5
    if cc.startswith('LA'):                 return 1
    if ct.startswith(('Mi', 'Tm', 'C2', 'C3', 'T1', 'T2', 'T3')): return 2  # medulla
    if cc.startswith('ME'):                 return 2
    if ct.startswith(('T4', 'T5')):         return 3     # motion detectors
    if cc.startswith(('LO', 'LOP')):        return 4     # lobula / lobula plate
    if sc == 'visual_projection':           return 5     # to central brain
    if sc == 'visual_centrifugal':          return 6     # feedback
    return 3                                              # default mid

# per-type depth (use the modal annotation row per type)
type_info = (vis.dropna(subset=['cell_type'])
                .groupby('cell_type')
                .agg(super_class=('super_class', lambda s: s.mode().iat[0]),
                     cell_class=('cell_class', lambda s: s.mode().iat[0] if s.notna().any() else 'nan'))
                .reset_index())
type_info['depth'] = type_info.apply(type_depth, axis=1)
depth = type_info.set_index('cell_type')['depth'].reindex(types_all).fillna(3).astype(int)

order = np.argsort(depth.to_numpy(), kind='stable')
ordered_types = [types_all[i] for i in order]
# show the top-K populous, but in depth order
top_set = set(top_types)
sel_ord = [i for i in order if types_all[i] in top_set]
lab = [types_all[i] for i in sel_ord]

fig, ax = plt.subplots(figsize=(12, 10))
M = np.log10(T_raw[np.ix_(sel_ord, sel_ord)] + 1)
sns.heatmap(M, ax=ax, cmap='magma', xticklabels=lab, yticklabels=lab,
            cbar_kws={'label': 'log10(syn+1)'})
ax.set(title='Type->type, reordered by feedforward depth (retina top-left -> VPN bottom-right)',
       xlabel='postsynaptic (deeper ->)', ylabel='presynaptic (deeper ->)')
ax.tick_params(labelsize=6)
plt.tight_layout(); plt.show()

# feedforward vs feedback synapse mass (full matrix)
d = depth.to_numpy()
pre_d = d[:, None]; post_d = d[None, :]
ff = T_raw[(post_d > pre_d)].sum(); fb = T_raw[(post_d < pre_d)].sum(); lat = T_raw[(post_d == pre_d)].sum()
tot = ff + fb + lat
print(f"feedforward (deeper) mass: {ff/tot:.0%}   feedback: {fb/tot:.0%}   lateral/same-depth: {lat/tot:.0%}")

## D. Input → output reachability

Can a signal from the photoreceptors actually reach the readout types, and how deep is
the path? BFS on the (thresholded) type graph from the input set; report the depth at
which each readout (VPN) type is first reached, and flag any types unreachable from the
retina.

In [ ]:
import collections
# adjacency at the type level (binary, presence of a type->type connection)
adj = {t: set() for t in types_all}
nz = np.argwhere(T_raw > 0)
for a, b in nz:
    adj[types_all[a]].add(types_all[b])

def bfs_depths(sources):
    dist = {s: 0 for s in sources if s in adj}
    q = collections.deque(dist)
    while q:
        u = q.popleft()
        for v in adj[u]:
            if v not in dist:
                dist[v] = dist[u] + 1; q.append(v)
    return dist

reach = bfs_depths(photo_types)
print(f"types reachable from photoreceptors: {len(reach)}/{len(types_all)}")
unreached = [t for t in types_all if t not in reach]
print(f"unreachable types: {len(unreached)} (e.g. {unreached[:10]})")

# how deep are the VPN readout types?
vpn_reach = {t: reach[t] for t in vpn_types if t in reach}
if vpn_reach:
    import numpy as _np
    ds = _np.array(list(vpn_reach.values()))
    print(f"\nVPN readout types reached: {len(vpn_reach)}/{len(vpn_types)}  "
          f"depth min/median/max = {ds.min()}/{int(_np.median(ds))}/{ds.max()}")
    sns.histplot(ds, bins=range(0, ds.max()+2), discrete=True)
    plt.xlabel('BFS depth from photoreceptors'); plt.ylabel('# VPN types')
    plt.title('How many synaptic hops from retina to each readout (VPN) type')
    plt.show()

## E. Per-type signal properties

For each type: type-level in/out degree (how many other types it talks to), incoming /
outgoing synapse mass, dominant neurotransmitter (→ its **row sign** in the weight
matrix), and depth. This is the per-node identity table the model uses to set each
node's sign and scale.

In [ ]:
nt_by_type = (vis.dropna(subset=['cell_type'])
                 .groupby('cell_type')['nt_canonical']
                 .agg(lambda s: s.mode().iat[0] if s.notna().any() else None))
type_tbl = pd.DataFrame({
    'type': types_all,
    'n_neurons': counts.reindex(types_all).to_numpy(),
    'depth': depth.reindex(types_all).to_numpy(),
    'out_types': (T_raw > 0).sum(axis=1),
    'in_types': (T_raw > 0).sum(axis=0),
    'out_syn': T_raw.sum(axis=1),
    'in_syn': T_raw.sum(axis=0),
    'nt': [nt_by_type.get(t) for t in types_all],
}).set_index('type')
print("most-connected visual types (by out_syn):")
display(type_tbl.sort_values('out_syn', ascending=False).head(15))
print("\nphotoreceptor + sample VPN rows:")
display(type_tbl.loc[[t for t in photo_types + vpn_types[:5] if t in type_tbl.index]])

## F. Sanity vs known biology

Cheap correctness checks before modeling. Known facts: photoreceptors are
**cholinergic / histaminergic input** with ~no input from other visual types (they're
the source); **L1→Mi1 (ON)** and **L2→Tm (OFF)** pathways feed the motion detectors;
**T4 reads the medulla, T5 reads the lobula**. We verify a few of these in the collapsed
graph.

In [ ]:
def syn(a, b):
    if a in ti and b in ti: return T_raw[ti[a], ti[b]]
    return float('nan')

checks = [
    ('R1-6 -> L1 (lamina input)',          syn('R1-6', 'L1')),
    ('R1-6 -> L2',                          syn('R1-6', 'L2')),
    ('L1 -> Mi1 (ON pathway)',              syn('L1', 'Mi1')),
    ('L2 -> Tm2 (OFF pathway)',             syn('L2', 'Tm2')),
    ('Mi1 -> T4c (ON motion)',              syn('Mi1', 'T4c')),
    ('Tm1 -> T5c (OFF motion)',             syn('Tm1', 'T5c')),
    ('photoreceptor in-syn from visual types (small; lateral/feedback only)',
        sum(T_raw[:, ti[p]].sum() for p in photo_types if p in ti)),
    ('   ...vs photoreceptor OUT-syn (should dominate; they are the input)',
        sum(T_raw[ti[p], :].sum() for p in photo_types if p in ti)),
]
for name, v in checks:
    print(f"  {name:62s} : {v:,.0f}")
# All forward pathways above should be large & positive; photoreceptor in << out.

## G. Modeling-readiness summary

**Resulting first-cut model spec (cell-type level):**

- **Nodes** = the visual `cell_type`s present above (~680; prune to a working set, e.g.
  the populous optic types + chosen VPN readouts).
- **Weight matrix** `W` = signed, per-source-normalized type→type matrix. The matrix here
  is source-major (`T[a,b] = a→b`); for a rate update `r_next = f(W @ r)` use `W = (T_norm * sign).T`.
- **Input nodes** = photoreceptor types `R1-6, R7, R8` — drive these with the (downsampled)
  image; everything else evolves by the recurrent/feedforward weights.
- **Readout nodes** = visual-projection types (`LC*`, `MeTu*`, …) → linear head to class labels.
- **Node identity** (sign, τ scale) from the per-type table in section E.

**Open decisions to make before stage 3:**
1. `cell_type` vs `cell_type`+`side` as the node key (1350 nodes if split L/R).
2. Which `super_class`es to include (optic only? + VPN? + centrifugal feedback?).
3. Normalization (per-source-neuron, as here, vs per-target-neuron vs none) and whether to
   keep monoamine (sign-0) edges.
4. Rate (leaky-integrator, flyvis-style) vs plain feedforward MLP-on-the-graph for v1.

**Known data gap (only matters if going flyvis-faithful later):** no column/retinotopy
assignment in `neurons.parquet` — needed for per-column hexagonal input + weight sharing
across columns. Closeable via the FlyWire optic-lobe columnar annotations or by deriving
columns from `soma_x/y/z`. Not required for this cell-type-level model.